# 07 — Appendix: merge the adapter, export to GGUF, use llama.cpp

**Chapter 3** · appendix / homework · GPU recommended · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/en/chapter_3/07_appendix_merge_gguf.ipynb)

![Chapter 3 roadmap](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_3/figures/00_roadmap.png)

At the end of notebook 05, we have **two** artifacts: the base model (14.5 GB, on Hugging Face) and our **adapter** (~160 MB). To deploy the model outside Python (on a laptop, in an application), we often use **llama.cpp** and its **GGUF** file format. Two paths:

| Path | What we produce | Resources | Benefit |
|---|---|---|---|
| **A. Adapter → GGUF** | a small GGUF adapter file, applied to an existing GGUF base | light (CPU, a few minutes) | keep modularity: one base, several adapters |
| **B. Merge → GGUF** | a single GGUF model, adapter built in | ~16 GB of RAM or a GPU ≥ 24 GB, ~30 GB of disk | a single file, no inference overhead |

> **Status:** the merge (§1) was tested on a small model with the same architecture. The llama.cpp commands (§2 and §3) follow the project's documentation but were not run end to end on the 7B: check them before class, llama.cpp evolves fast.

In [ ]:
# 0. Installation (Colab) — versions tested together; ~1 min
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.1.0"
    if not os.path.exists("finetune.py"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
        %cd ai-masterclass/en/chapter_3

import math, json, time, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
import matplotlib.pyplot as plt
import finetune as ft

assert torch.cuda.is_available(), "No GPU: on Colab, Runtime > Change runtime type > GPU (T4)."
GPU = torch.cuda.get_device_name(0)
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere (A100, L4…) and newer; not the T4
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {GPU} ({GPU_MEMORY:.1f} GB) | compute in {'BF16' if BF16 else 'FP16 (no BF16 on this GPU)'}")
print("PyTorch", torch.__version__)

In [ ]:
CORPUS = "kaamelott"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_DIR = f"adapters/mistral7b-{CORPUS}-qlora"
DO_MERGE = True                # §1: needs ~16 GB of RAM (Colab "High-RAM") or a GPU >= 24 GB
DO_GGUF = True                 # §2: clones llama.cpp, converts the adapter, downloads a GGUF base (4.4 GB)

from pathlib import Path
assert Path(ADAPTER_DIR).exists(), "Run notebook 05 first (or copy the adapter back from Google Drive)."

## 1. Merge the adapter into the model (`merge_and_unload`)

This is the merge from notebook 03, §5, applied to 7 billion parameters: each targeted layer becomes `W₀ + (α/r)·BA`. **Important:** we merge into a **16-bit** base, not into the 4-bit base used for training: otherwise we would write `ΔW` into a quantized matrix and lose most of the correction.

In [ ]:
if DO_MERGE:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    on_gpu = GPU_MEMORY >= 20                     # the FP16 model (14.5 GB) fits on an L4 / A100, not on a T4
    print("merging on", "GPU" if on_gpu else "CPU (RAM)")
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16,
                                                device_map={"": 0} if on_gpu else "cpu", low_cpu_mem_usage=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)

    # check: merging does not change the outputs (up to FP16 rounding)
    test = tokenizer("ARTHUR: Bon, on attaque ?", return_tensors="pt").to(base.device)
    with torch.no_grad():
        before = model(**test).logits.float()
    merged = model.merge_and_unload()
    with torch.no_grad():
        after = merged(**test).logits.float()
    print(f"max logit difference before/after merging: {(before - after).abs().max().item():.3f} "
          f"(order of magnitude of the logits: {before.abs().max().item():.1f})")
    print("any LoRA layers left?", any("lora" in n for n, _ in merged.named_modules()))

    MERGED_DIR = f"merged/mistral7b-{CORPUS}"
    merged.save_pretrained(MERGED_DIR, max_shard_size="2GB")
    tokenizer.save_pretrained(MERGED_DIR)
    size = sum(f.stat().st_size for f in Path(MERGED_DIR).iterdir()) / 1024**3
    print(f"merged model: {MERGED_DIR} ({size:.1f} GB)")
    del base, model, merged
    torch.cuda.empty_cache()

The `merged/...` folder is an ordinary Hugging Face model: `AutoModelForCausalLM.from_pretrained(MERGED_DIR)` loads it without `peft`. We lost modularity (one model per style) but gained simplicity.

## 2. Path A: the adapter as GGUF, applied to a GGUF base

1. `convert_lora_to_gguf.py` (llama.cpp) reads `adapter_config.json`, fetches the base model's configuration from Hugging Face, and writes the adapter in GGUF format;
2. we download an **already quantized** GGUF base (Q4_K_M, 4.4 GB);
3. `llama-cpp-python` loads the base and applies the adapter at inference time.

Note: the adapter was trained on an NF4 base and will be applied to a Q4_K_M base. The two quantizations differ slightly; in practice the effect is small, but it is a source of discrepancy worth measuring.

In [ ]:
if DO_GGUF:
    if not Path("llama.cpp").exists():
        !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    %pip install -q -e llama.cpp/gguf-py
    !python llama.cpp/convert_lora_to_gguf.py {ADAPTER_DIR} --outtype f16 --outfile adapters/{CORPUS}-lora-f16.gguf
    !ls -lh adapters/*.gguf

In [ ]:
if DO_GGUF:
    from huggingface_hub import hf_hub_download
    base_gguf = hf_hub_download("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf")
    print(base_gguf)
    # llama-cpp-python is compiled at install time: ~5 min for the CPU version.
    # GPU version (longer to compile): CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python
    %pip install -q llama-cpp-python

In [ ]:
if DO_GGUF:
    from llama_cpp import Llama
    prompt = [{"role": "user", "content": f"{ft.INSTRUCTIONS[CORPUS]}\n\nARTHUR: Bon. Quelqu’un a une idée pour récupérer le Graal avant ce soir ?"}]
    for lora in [None, f"adapters/{CORPUS}-lora-f16.gguf"]:
        llm = Llama(model_path=base_gguf, lora_path=lora, n_ctx=1024, n_gpu_layers=0, seed=0, verbose=False)
        output = llm.create_chat_completion(messages=prompt, max_tokens=80, temperature=0.7)
        print("with adapter" if lora else "base only   ", ":", output["choices"][0]["message"]["content"])
        del llm

On the command line, with the llama.cpp binaries, this is the `--lora` option:

```bash
llama-cli -m Mistral-7B-Instruct-v0.3-Q4_K_M.gguf --lora kaamelott-lora-f16.gguf -cnv
```

## 3. Path B: the merged model as GGUF (going further)

Starting from the `merged/...` folder from §1 (≈ 30 GB of disk in total):

```bash
# 1. conversion Hugging Face -> 16-bit GGUF (~14.5 GB)
python llama.cpp/convert_hf_to_gguf.py merged/mistral7b-kaamelott --outtype f16 --outfile kaamelott-f16.gguf
# 2. build llama.cpp (once), then quantize to 4 bits (~4.4 GB)
cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --target llama-quantize llama-cli -j
llama.cpp/build/bin/llama-quantize kaamelott-f16.gguf kaamelott-Q4_K_M.gguf Q4_K_M
# 3. usage
llama.cpp/build/bin/llama-cli -m kaamelott-Q4_K_M.gguf -cnv
```

The `kaamelott-Q4_K_M.gguf` file works in every GGUF-compatible tool (llama.cpp, Ollama, LM Studio…), on a laptop without a GPU.

## License and redistribution

Before **publishing** an adapter or a merged model, distinguish:
- the license of the **Mistral weights**: Apache 2.0, redistribution allowed;
- the rights on the **data**: the *Kaamelott* dialogues are a copyrighted work; a transcript available online is not royalty-free. An adapter that **memorizes** lines (notebook 06) redistributes part of the work. Plato (Cousin's translation, 19th c.) and Shakespeare are in the public domain.

"Downloadable" does not mean "freely redistributable".

## Summary

- `merge_and_unload()` writes `W₀ + (α/r)·BA` into each layer: **same model**, no more dependency on `peft`, no inference overhead. Always merge into a **16-bit** base.
- **GGUF** is llama.cpp's format; you can convert **the adapter alone** (light, modular path) or the **merged model** and then quantize it (single file).
- Publishing a fine-tuned model involves the rights on the **data**, not just the base model's license.